# Selector Lab: When Does Best-of-N Help?

**Authors:** Alicia Chua, Pawarit Laosunthara, and Eric Tang, Anyscale

This hardware-free notebook studies the selector behind the video audition. It uses the same eight illustrative ratings as the interactive lesson. No video model or GPU is required.

By the end, you will be able to:

1. compute a Best-of-N ranking,
2. test how mission weights change the winner,
3. measure how sensitive a selection is to rating uncertainty, and
4. explain why a larger candidate pool does not fix a weak selector.

**Scope:** The ratings are fixed illustrative visual annotations. They are not ground truth, benchmark measurements, or autonomous-vehicle safety metrics.

## 1. Load the teaching data

Run this notebook from the teaching package root or from the `notebooks` directory. The helper below finds the score CSV in either location.

In [ ]:
from pathlib import Path
import csv
import random
from collections import Counter

candidates = [
    Path('data/rubric_scores.csv'),
    Path('../data/rubric_scores.csv'),
]
data_path = next((path for path in candidates if path.exists()), None)
if data_path is None:
    raise FileNotFoundError('Run from the package root or notebooks directory.')

with data_path.open(newline='', encoding='utf-8') as handle:
    rows = list(csv.DictReader(handle))

for row in rows:
    row['seed'] = int(row['seed'])
    for key in ['weather_fidelity', 'structure_preservation', 'temporal_stability']:
        row[key] = float(row[key])

print(f'Loaded {len(rows)} candidates from {data_path}')
print('Fields:', ', '.join(rows[0]))

## 2. Write the decision rule

For an eligible candidate, the lesson uses a normalized weighted average. Hard rejection gates should be applied before this calculation.

In [ ]:
FIELDS = {
    'weather': 'weather_fidelity',
    'structure': 'structure_preservation',
    'stability': 'temporal_stability',
}

def score(row, weights):
    total = sum(weights.values())
    if total <= 0:
        raise ValueError('At least one weight must be positive.')
    return sum(weights[name] * row[field] for name, field in FIELDS.items()) / total

def rank(pool, weights, rejected=()):
    rejected = set(rejected)
    eligible = [row for row in pool if row['candidate'] not in rejected]
    if not eligible:
        return []
    return sorted(
        [(row['candidate'], score(row, weights)) for row in eligible],
        key=lambda item: (-item[1], item[0]),
    )

balanced = {'weather': 40, 'structure': 35, 'stability': 25}
rank(rows, balanced)[:3]

### Checkpoint

Before running the next cell, predict the selected take for N = 1, 2, 4, and 8. Remember that these are nested pools.

In [ ]:
for n in [1, 2, 4, 8]:
    ranking = rank(rows[:n], balanced)
    winner, winner_score = ranking[0]
    runner_up = ranking[1][0] if len(ranking) > 1 else 'none'
    print(f'N={n}: winner={winner}, score={winner_score:.1f}, runner-up={runner_up}')

A larger N changes the available pool. It does not promise that the selected result will change, nor that the selector is correct.

**Discuss:** What evidence would justify the extra generation and review cost between N = 4 and N = 8?

## 3. Change the mission

The same candidates can support different decisions because a selector encodes a purpose. Compare balanced, structure-first, and weather-only profiles.

In [ ]:
profiles = {
    'balanced': {'weather': 40, 'structure': 35, 'stability': 25},
    'structure_first': {'weather': 5, 'structure': 55, 'stability': 40},
    'weather_only': {'weather': 100, 'structure': 0, 'stability': 0},
}

for name, weights in profiles.items():
    top = rank(rows, weights)[:3]
    print(name)
    for position, (candidate, value) in enumerate(top, start=1):
        print(f'  {position}. {candidate}: {value:.1f}')

The weather-only profile selects the strongest atmosphere even when structure and stability are weak. This is a visible proxy failure. A production selector should also apply hard gates. Try `rank(rows, profiles['weather_only'], rejected={'take07'})`.

## 4. Map weight sensitivity

The next cell checks every five-point weight combination that sums to 100. It counts how often each candidate wins. This is a simple way to see whether the decision is tied to one narrow mission profile.

In [ ]:
winner_counts = Counter()
examples = {}
for weather_weight in range(0, 101, 5):
    for structure_weight in range(0, 101 - weather_weight, 5):
        stability_weight = 100 - weather_weight - structure_weight
        weights = {
            'weather': weather_weight,
            'structure': structure_weight,
            'stability': stability_weight,
        }
        winner = rank(rows, weights)[0][0]
        winner_counts[winner] += 1
        examples.setdefault(winner, weights)

total_profiles = sum(winner_counts.values())
for candidate, count in winner_counts.most_common():
    print(f'{candidate}: {count / total_profiles:6.1%} of profiles; example={examples[candidate]}')

## 5. Add rating uncertainty

A point estimate can hide reviewer disagreement. The simulation below perturbs each dimension with Gaussian noise. The standard deviation is a teaching assumption, not an empirical uncertainty estimate.

In [ ]:
def selection_probabilities(pool, weights, rating_sd=5.0, draws=5000, seed=7):
    rng = random.Random(seed)
    wins = Counter()
    for _ in range(draws):
        perturbed = []
        for row in pool:
            copy = dict(row)
            for field in FIELDS.values():
                copy[field] = min(100, max(0, rng.gauss(row[field], rating_sd)))
            perturbed.append(copy)
        wins[rank(perturbed, weights)[0][0]] += 1
    return {name: count / draws for name, count in wins.most_common()}

selection_probabilities(rows, balanced)

### Interpret carefully

If the top candidate has a modest selection probability, the point-estimate winner is not robust to plausible rating changes. The correct response may be more review, a better rubric, or abstention. It is not automatically a larger N.

## 6. Transfer challenge

Choose another generative task. Define:

- three operational criteria,
- one hard rejection gate,
- one human-calibration step,
- one automated evidence source, and
- one reason to abstain.

Then answer: What could your selector reward by accident?